In [42]:
from dotenv import load_dotenv
from lmstudio import get_lmstudio_client

load_dotenv()

openai_client = get_lmstudio_client()



In [43]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
    )
    return response.output_text

In [3]:
question = "I just discovered the course. Can I join now?"

In [4]:
from ingest import load_faq_data, build_index
index = build_index(load_faq_data())


In [5]:
INSTRUCTIONS = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

'''

In [6]:
def search_index(question):
    return index.search(question, 
    boost_dict={"question": 3.0, "answer": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},

    num_results=5)

In [7]:
def build_prompt(question, search_results):
    return f'''
    {INSTRUCTIONS}

    
    Question: {question}

    Context:
    {search_results}
    '''

In [8]:
def build_context(search_results):

    lines = []
    for doc in search_results:
        lines.append(doc['section'])
        lines.append("Q: " + doc['question'])
        lines.append("A: " + doc['answer'])
        lines.append("")
    return '\n'.join(lines).strip()



In [9]:
def rag(question):
    search_results = search_index(question)
    user_prompt = build_prompt(question, build_context(search_results))    
    return llm(user_prompt)

In [10]:
res = rag(question)

In [11]:
res

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [12]:
context = build_context(search_index(question))
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [16]:
prompt = build_prompt(question, context)

In [51]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=prompt,
)


In [29]:
response.output_text

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [30]:
response.output[0].content[0].text

'\n*   Task: Answer a question based on the provided context.\n    *   Constraint 1: Use the context to find relevant information.\n    *   Constraint 2: Provide accurate answers.\n    *   Constraint 3: If the answer isn\'t in the context, respond with "I don\'t know."\n    *   User Question: "I just discovered the course. Can I join now?"\n    *   Context provided: A set of FAQs about a course (LLM Zoomcamp).\n\n    *   Question 1 in context: "Q: I just discovered the course. Can I still join? A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions."\n\n    *   The user\'s question ("Can I join now?") matches the first question in the context ("Can I still join?").\n    *   The answer is: "Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions."\n\n    *   The answer is directly in the context.\n    *   Formulate the final response.'

In [52]:
response.usage

ResponseUsage(input_tokens=427, input_tokens_details=InputTokensDetails(cached_tokens=422), output_tokens=237, output_tokens_details=OutputTokensDetails(reasoning_tokens=209), total_tokens=664)

In [36]:
def llm(instructions, user_prompt, model="gpt-4o-mini"):
    message_history = [
        {"role": "developer", "content": instructions}, #constant
        {"role": "user", "content": user_prompt}, #variable
    ]


    response = openai_client.responses.create(
        model="gpt-4o-mini",
        input=message_history,
    )
    return response.output_text
    
    

In [37]:
def rag(query, model="gpt-4o-mini"):
    search_results = search_index(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer



In [39]:
answer = rag(question)
print(answer)

You can join the course now. However, if you want to receive a certificate, you need to submit your project while the submissions are still being accepted.


In [40]:
from rag_helper import RAGBase

rag_helper = RAGBase(
    index=index,
    llm_client=openai_client,
    model="gpt-4o-mini",
)

In [41]:
rag_helper.rag(question)

"Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while we're still accepting submissions."